## Run jobs in a batch
https://quantum.cloud.ibm.com/docs/en/ibm-quantum-compute/run-jobs-batch

In [1]:
from qiskit_ibm_runtime import QiskitRuntimeService, Batch, Executor

# Legacy server-side primitive before qiskit-ibm-runtime 0.50.0
# from qiskit_ibm_runtime import Sampler, Estimator

# Client-side primitive introduced in qiskit-ibm-runtime 0.50.0
from qiskit_ibm_runtime.executor_sampler import Sampler
from qiskit_ibm_runtime.executor_estimator import Estimator

service = QiskitRuntimeService()

qiskit_runtime_service.__init__:WARNING:2026-11-16 07:44:23,879: Instance was not set at service instantiation. Free and trial plan instances will be prioritized. Based on the following filters: (tags: None, region: us-east, eu-de), and available plans: (open), the available account instances are: surnandi. If you need a specific instance set it explicitly either by using a saved account with a saved default instance or passing it in directly to QiskitRuntimeService(). Alternatively, pass instance='auto' or save it to your account for auto-selection without warning.


In [2]:
# Open a batch

backend = service.least_busy(operational=True, simulator=False)
batch = Batch(backend=backend)
estimator = Estimator(mode=batch)
sampler = Sampler(mode=batch)
executor = Executor(mode=batch)
# Close the batch because no context manager was used.
batch.close()

qiskit_runtime_service.backends:WARNING:2026-11-16 07:46:43,991: Loading instance: surnandi, plan: open
qiskit_runtime_service.backends:WARNING:2026-11-16 07:46:47,251: Using instance: surnandi, plan: open


In [4]:
# Context manager

from qiskit_ibm_runtime import Batch, Executor

# Legacy server-side primitive before qiskit-ibm-runtime 0.50.0
# from qiskit_ibm_runtime import Sampler, Estimator

# Client-side primitive introduced in qiskit-ibm-runtime 0.50.0
from qiskit_ibm_runtime.executor_sampler import Sampler
from qiskit_ibm_runtime.executor_estimator import Estimator

backend = service.least_busy(operational=True, simulator=False)
with Batch(backend=backend):
    estimator = Estimator()
    sampler = Sampler()
    executor = Executor()

qiskit_runtime_service.backends:WARNING:2026-11-16 07:47:51,727: Loading instance: surnandi, plan: open
qiskit_runtime_service.backends:WARNING:2026-11-16 07:47:53,221: Using instance: surnandi, plan: open


In [8]:
# batch length

from qiskit_ibm_runtime.executor_estimator import Estimator
with Batch(backend=backend, max_time="25m"):
    estimator = Estimator()
    sampler = Sampler()
    executor = Executor()
    

In [10]:
# Close a batch

with Batch(backend=backend) as batch:
    estimator = Estimator() 
    sampler = Sampler()
    job1 = estimator.run([estimator_pub])
    job2 = sampler.run([sampler_pub])

# The batch is no longer accepting jobs but the submitted job will run to completion.
result = job1.result()
result2 = job2.result()

NameError: name 'estimator_pub' is not defined

In [11]:
batch = Batch(backend=backend)

# If using qiskit-ibm-runtime earlier than 0.24.0, change `mode=` to `batch=`
estimator = Estimator(mode=batch)
sampler = Sampler(mode=batch)
job1 = estimator.run([estimator_pub])
job2 = sampler.run([sampler_pub])
print(f"Result1: {job1.result()}")
print(f"Result2: {job2.result()}")

# Manually close the batch. Running and queued jobs will run to completion.
batch.close()

NameError: name 'estimator_pub' is not defined

In [12]:
# Determine batch details

from qiskit_ibm_runtime import QiskitRuntimeService, Batch

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

with Batch(backend=backend) as batch:
    print(batch.details())

qiskit_runtime_service.__init__:WARNING:2026-11-16 07:54:21,344: Instance was not set at service instantiation. Free and trial plan instances will be prioritized. Based on the following filters: (tags: None, region: us-east, eu-de), and available plans: (open), the available account instances are: surnandi. If you need a specific instance set it explicitly either by using a saved account with a saved default instance or passing it in directly to QiskitRuntimeService(). Alternatively, pass instance='auto' or save it to your account for auto-selection without warning.
qiskit_runtime_service.backends:WARNING:2026-11-16 07:54:21,955: Loading instance: surnandi, plan: open
qiskit_runtime_service.backends:WARNING:2026-11-16 07:54:24,700: Using instance: surnandi, plan: open


{'id': '20ec6d0b-7dbb-4ca0-92f3-5e7a28a07e7f', 'backend_name': 'ibm_marrakesh', 'interactive_timeout': 1, 'max_time': 600, 'active_timeout': 600, 'state': 'open', 'accepting_jobs': True, 'last_job_started': None, 'last_job_completed': None, 'started_at': None, 'closed_at': None, 'activated_at': None, 'mode': 'batch', 'usage_time': None, 'timestamps': [{'status': 'open', 'timestamp': '2026-11-16T02:24:25.142Z'}], 'usage': None}


In [13]:
# Reconfigure jobs for parallel processing

from qiskit.circuit.random import random_circuit
from qiskit_ibm_runtime import Batch

# Legacy server-side primitive before qiskit-ibm-runtime 0.50.0
# from qiskit_ibm_runtime import Sampler

# Client-side primitive introduced in qiskit-ibm-runtime 0.50.0
from qiskit_ibm_runtime.executor_sampler import Sampler

max_circuits = 100
circuits = [pm.run(random_circuit(5, 5)) for _ in range(5 * max_circuits)]
for circuit in circuits:
    circuit.measure_active()
all_partitioned_circuits = []
for i in range(0, len(circuits), max_circuits):
    all_partitioned_circuits.append(circuits[i : i + max_circuits])
jobs = []
start_idx = 0

with Batch(backend=backend):
    sampler = Sampler()
    for partitioned_circuits in all_partitioned_circuits:
        job = sampler.run(partitioned_circuits)
        jobs.append(job)

NameError: name 'pm' is not defined